In [0]:
CREATE OR REPLACE TEMPORARY VIEW MPSII_Treatment_Table AS

SELECT *
FROM (
    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        NDC11 AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('54092070001','540920700')

    UNION

    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        NDC11 AS CODE,
        PHARMACY_EVENT_ID AS EVENT_ID,
        FILL_DATE,
        NULL AS PLACE_OF_SERVICE,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        'PHARMACY_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'

    UNION

    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        RENDERING_NPI AS NPI,
        PROCEDURE_CODE AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN (
        '99601','99602','96365','96366',
        'J1743','S9357','S9379',
        '38206','38230','38232',
        '38240','38241','38242',
        '38243','38250'
    )

)

WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30';

In [0]:
CREATE OR REPLACE TEMPORARY VIEW mpsii_diagnosis_table AS
WITH mpsii_1dx_specified AS (

    SELECT *
    FROM (
        SELECT DISTINCT
            PATIENT_ID,
            COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
            SERVICE_DATE AS FILL_DATE,
            MEDICAL_EVENT_ID AS EVENT_ID,
            DIAGNOSIS_CODES,
            KH_PLAN_ID AS KH_PLAN,
            PLACE_OF_SERVICE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE DIAGNOSIS_CODES LIKE '%E761%'

        UNION

        SELECT DISTINCT
            PATIENT_ID,
            PRESCRIBER_NPI AS NPI,
            FILL_DATE,
            PHARMACY_EVENT_ID AS EVENT_ID,
            DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
            COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
            NULL AS PLACE_OF_SERVICE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE = 'E761'
          AND TRANSACTION_STATUS = 'PAID'

    ) AS combined

    WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30'
),

mpsii_2dx_specified AS (
    SELECT *
    FROM mpsii_1dx_specified
    WHERE patient_id IN (
        SELECT a.patient_id
        FROM mpsii_1dx_specified AS a
        GROUP BY a.patient_id
        HAVING COUNT(DISTINCT a.fill_date) >= 2
    )
),

mpsii_1dx_unspecified AS (

    SELECT *
    FROM (
        SELECT DISTINCT
            PATIENT_ID,
            COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
            SERVICE_DATE AS FILL_DATE,
            MEDICAL_EVENT_ID AS EVENT_ID,
            DIAGNOSIS_CODES,
            KH_PLAN_ID AS KH_PLAN,
            PLACE_OF_SERVICE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE DIAGNOSIS_CODES LIKE '%E763%'

        UNION

        SELECT DISTINCT
            PATIENT_ID,
            PRESCRIBER_NPI AS NPI,
            FILL_DATE,
            PHARMACY_EVENT_ID AS EVENT_ID,
            DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
            COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
            NULL AS PLACE_OF_SERVICE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE = 'E763'
          AND TRANSACTION_STATUS = 'PAID'

    ) AS combined


    WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-11-30'
),

mpsii_2dx_unspecified AS (
    SELECT *
    FROM mpsii_1dx_unspecified
    WHERE patient_id IN (
        SELECT a.patient_id
        FROM mpsii_1dx_unspecified AS a
        GROUP BY a.patient_id
        HAVING COUNT(DISTINCT a.fill_date) >= 2
    )
),

mpsii_2dx_specified_tx AS (
    SELECT *
    FROM mpsii_treatment_table
    WHERE patient_id IN (
        SELECT DISTINCT patient_id
        FROM mpsii_2dx_specified
    )
),

incremental_patient AS (
    SELECT DISTINCT patient_id
    FROM mpsii_2dx_unspecified
    WHERE patient_id IN (
        SELECT DISTINCT patient_id
        FROM mpsii_treatment_table
        WHERE code IN ('54092070001','540920700','J1743')
    )
    AND patient_id NOT IN (
        SELECT DISTINCT patient_id
        FROM mpsii_2dx_specified_tx
    )
),

all_dx_patients_claims AS (
    SELECT *
    FROM mpsii_2dx_specified

    UNION

    SELECT *
    FROM mpsii_1dx_unspecified
    WHERE patient_id IN (
        SELECT patient_id
        FROM incremental_patient
    )
)

SELECT *
FROM all_dx_patients_claims;

In [0]:
select * from mpsii_diagnosis_table

In [0]:
create or replace temp view all_tx_patient_count as 
with tx_patient as (
  select b.hco_npi, count(distinct a.patient_id)
  from MPSII_Treatment_Table a 
  left join com_edp_prd.cmpa_insights_internal_schema.reference_file_0109 b on a.npi = b.hcp_npi
  where b.hco_npi is not null and b.hco_npi <> '-'
  group by b.hco_npi
  sort by 2
)
select * from tx_patient

In [0]:
select * from all_tx_patient_count

In [0]:
create or replace temp view all_dx_patient_count as 
with dx_patient as (
  select b.hco_npi, count(distinct a.patient_id)
  from mpsii_diagnosis_table a 
  left join com_edp_prd.cmpa_insights_internal_schema.reference_file_0109 b on a.npi = b.hcp_npi
  where b.hco_npi is not null and b.hco_npi <> '-'
  group by b.hco_npi
  sort by 2
)
select * from dx_patient

In [0]:
select * from all_dx_patient_count

In [0]:
create or replace temp view tx_specialty as
select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from mpsii_treatment_table a 
    left join com_edp_prd.com_raw.kom_providers b
on a.npi = b.npi
;

In [0]:
Create or replace temp view tx_primary_hcp AS
SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID AS N_PATS, 
    Fill_date,
    'DX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date, 
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM tx_specialty
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE
                    )
                )
            )
        ) 
    ) 
    WHERE HCP_RANK = 1
);


In [0]:
create or replace temp view tx_unique_count as 
select b.hco_npi, count(distinct a.n_pats)
from tx_primary_hcp a 
left join com_edp_prd.cmpa_insights_internal_schema.reference_file_0109 b
on a.NPI = b.hcp_npi
where b.hco_npi is not null and b.hco_npi <> '-'
group by 1 order by 2

In [0]:
select * from tx_unique_count

In [0]:
create or replace temp view dx_specialty as
select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from mpsii_diagnosis_table a 
    left join com_edp_prd.com_raw.kom_providers b
on a.npi = b.npi
;

In [0]:
Create or replace temp view dx_primary_hcp AS
SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID AS N_PATS, 
    Fill_date,
    'DX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date, 
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM dx_specialty
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE
                    )
                )
            )
        ) 
    ) 
    WHERE HCP_RANK = 1
);


In [0]:
create or replace temp view dx_unique_count as 
select b.hco_npi, count(distinct a.n_pats)
from dx_primary_hcp a 
left join com_edp_prd.cmpa_insights_internal_schema.reference_file_0109 b
on a.NPI = b.hcp_npi
where b.hco_npi is not null and b.hco_npi <> '-'
group by 1 order by 2

In [0]:
select * from dx_unique_count

In [0]:
create or replace temp view mpsii_dx_tx_patients as 
select *
from mpsii_diagnosis_table 
where patient_id in (select PATIENT_ID from mpsii_treatment_table)

In [0]:
select * from mpsii_dx_tx_patients

In [0]:
with all_claims as (
  SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE, 'Dx' as claim_type, KH_PLAN_ID as plan_id
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
UNION
SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE, 'Dx' as claim_type, coalesce(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) as plan_id
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761','E763')
  AND TRANSACTION_STATUS = 'PAID'
UNION
SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE, 'Tx' as claim_type, KH_PLAN_ID as plan_id
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('54092070001','540920700')
UNION
SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE, 'Tx' as claim_type, coalesce(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) as plan_id
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('54092070001','540920700')
  AND TRANSACTION_RESULT = 'PAID'
UNION
SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE, 'Tx' as claim_type, KH_PLAN_ID as plan_id
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                          '38206','38230','38232','38240','38241','38242','38243','38250')
),
relevant_patients as (
  select *
  from all_claims
  where patient_id in ('Y9GW7HPB')
),
patient_geography AS (
  SELECT *
  FROM (
    SELECT *,
      ROW_NUMBER() OVER (
        PARTITION BY patient_id
        ORDER BY
          CASE WHEN valid_to_date > CURRENT_DATE() THEN 1 ELSE 2 END,
          valid_to_date DESC
      ) AS rn
    FROM com_edp_prd.com_raw.kom_patient_geography
  )
  WHERE rn = 1
),
pulling_relevant_info as (
  select a.patient_id, a.npi, a.fill_date, b.PATIENT_YOB, (year(current_date) - year(b.PATIENT_YOB)) as age, c.PATIENT_STATE, d.FIRST_NAME, d.LAST_NAME, concat(d.FIRST_NAME, ' ', d.LAST_NAME) as hcp_name, d.PRIMARY_SPECIALTY, d.SECONDARY_SPECIALTY, a.claim_type, e.PAYER_NAME, e.INSURANCE_GROUP
  from relevant_patients as a
  left join com_edp_prd.com_raw.kom_patient_demographics as b on a.patient_id = b.PATIENT_ID
  left join patient_geography as c on a.patient_id = c.PATIENT_ID
  left join com_edp_prd.com_raw.kom_providers as d on a.npi = d.npi and d.provider_type = 'INDIVIDUAL'
  left join com_edp_prd.com_raw.kom_plans as e on a.plan_id = e.KH_PLAN_ID
)
select * from pulling_relevant_info 
-- where patient_id = 'Y9GW7HPB'

In [0]:
select * from com_raw.kom_medical_events
where PATIENT_ID in ('Y9GW7HPB')

In [0]:
select * from com_raw.kom_pharmacy_events
where PATIENT_ID in ('Y9GW7HPB')

In [0]:
select * from cmpa_insights_internal_schema.patient360_master

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.reference_file_0109

In [0]:

WITH zip_input AS (
    SELECT '78207' AS zip UNION ALL
    SELECT '78229' UNION ALL
    SELECT '78411' UNION ALL
    SELECT '84107' UNION ALL
    SELECT '29646' UNION ALL
    SELECT '94118' UNION ALL
    SELECT '94612' UNION ALL
    SELECT '92503' UNION ALL
    SELECT '10032' UNION ALL
    SELECT '10034' UNION ALL
    SELECT '10709' UNION ALL
    SELECT '40536' UNION ALL
    SELECT '94583'
),

zip_to_region_territory AS (
    SELECT
        a.zip,
        b.territory_id,
        b.territory_name AS territory,
        b.region_id,
        b.region_name AS region,
        b.city,
        b.state
    FROM zip_input a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping b
        ON TRY_CAST(a.zip AS BIGINT) = b.zipcode
)
SELECT *
FROM zip_to_region_territory
ORDER BY region, territory, zip;
 